In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. NATIVE IMPORTS FROM YOUR PIPELINE ---
from config import SimConfig
from src.utils.data_loader import load_power_profile

config = SimConfig()

# --- 2. MARKOV CACHING ENGINE ---
# We cache the transition matrices so the dashboard doesn't re-parse 
# thousands of rows of telemetry every time you move a slider.
_MARKOV_CACHE = {}

def get_transition_matrix(training_day: int, p_vals: np.ndarray):
    """Loads telemetry, quantizes to p_vals, and builds the Markov matrix."""
    if training_day in _MARKOV_CACHE:
        return _MARKOV_CACHE[training_day]
    
    # Load raw demand profile
    P_d_train = load_power_profile(day=training_day)
    
    # Quantize continuous power to discrete p_vals grid
    state_indices = np.abs(P_d_train[:, None] - p_vals).argmin(axis=1)
    
    # Build transition counts
    N_p = len(p_vals)
    trans_mat = np.zeros((N_p, N_p))
    for t in range(len(state_indices) - 1):
        trans_mat[state_indices[t], state_indices[t+1]] += 1
        
    # Normalize to probabilities
    row_sums = trans_mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0  # Prevent div-by-zero for unvisited states
    trans_mat = trans_mat / row_sums
    
    _MARKOV_CACHE[training_day] = trans_mat
    return trans_mat

# --- 3. PHYSICS MATH ENGINE ---
def compute_dynamic_cost_matrices(p_vals, n_vals, k_fc_override, s_max_override):
    """Calculates A, B, C and the full cost matrix using config.py physics."""
    
    # Pull physical constants from your config
    k_h2 = config.PHYSICS.k_h2
    tau_fc = config.PHYSICS.tau_fc
    a0 = config.PHYSICS.a0
    a1 = config.PHYSICS.a1
    a2 = config.PHYSICS.a2
    alpha = config.PHYSICS.alpha_deg
    p_nom = config.PHYSICS.p_nom
    
    # The U-Shape Expansion Math
    A = (a2 * (k_h2 / 1000.0)) + (k_fc_override * alpha) / (3600.0 * tau_fc * (p_nom**2))
    B = (a1 * (k_h2 / 1000.0)) - (2.0 * k_fc_override * alpha) / (3600.0 * tau_fc * p_nom)
    C = (a0 * (k_h2 / 1000.0)) + (k_fc_override / (3600.0 * tau_fc)) * (1.0 + alpha)
    
    k_s = k_fc_override / s_max_override
    
    # Pre-compute the full C_o(n, P_d) matrix
    cost_matrix = np.zeros((len(p_vals), len(n_vals)))
    for l, p in enumerate(p_vals):
        for j, n in enumerate(n_vals):
            cost_matrix[l, j] = A * (p**2) / n + B * p + C * n
            
    return A, B, C, cost_matrix, k_s

def compute_expected_holding_times(trans_mat, cost_matrix, tolerance):
    """Computes E[T_hold] exactly like the ExpectedCostHeuristicBase class."""
    N_p, N_n = cost_matrix.shape
    E_t_hold = np.zeros((N_p, N_n))
    
    # 1. Map Comfort Zones (Tolerance logic)
    # is_comfortable[l, j] is True if module count j is in top `tolerance` cheapest for demand l
    is_comfortable = np.zeros((N_p, N_n), dtype=bool)
    for l in range(N_p):
        cheapest_indices = np.argsort(cost_matrix[l, :])[:tolerance]
        is_comfortable[l, cheapest_indices] = True

    # 2. Calculate Expected Times
    for l in range(N_p):
        for j in range(N_n):
            # Sum probability of transitioning to any demand k where current module j is still comfortable
            p_hold = np.sum(trans_mat[l, is_comfortable[:, j]])
            
            # Convert to seconds using config.Ts
            if p_hold >= 0.999:
                E_t_hold[l, j] = config.Ts * 1000  # Artificial ceiling for absorbing states
            else:
                E_t_hold[l, j] = config.Ts / (1.0 - p_hold)
                
    return E_t_hold

# --- 4. MASTER DASHBOARD RENDERER ---
def render_dashboard(p_d_eval, current_n, k_fc, s_max, training_day, tolerance, heuristic_mode):
    # Retrieve grids
    p_vals = config.p_vals
    n_vals = config.n_vals
    
    # Core calculations
    trans_mat = get_transition_matrix(training_day, p_vals)
    A, B, C, cost_matrix, k_s = compute_dynamic_cost_matrices(p_vals, n_vals, k_fc, s_max)
    E_t_hold_matrix = compute_expected_holding_times(trans_mat, cost_matrix, tolerance)
    
    # Find exact indices for single-point plots
    idx_p = np.abs(p_vals - p_d_eval).argmin()
    idx_n0 = np.abs(n_vals - current_n).argmin()
    actual_p_d = p_vals[idx_p]
    
    # Initialize figure
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            f"1. Cost 'Bowl' @ P_d={actual_p_d:.1f}kW", 
            f"2. Temporal Hurdle (n={current_n} \u2192 n*)", 
            "3. Hysteresis Decision Heatmap"
        ),
        specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "heatmap"}]]
    )
    
    # --- PLOT 1: The Shallow Bowl ---
    costs = cost_matrix[idx_p, :]
    n_opt_analytic = np.sqrt(A/C) * actual_p_d
    fig.add_trace(go.Scatter(x=n_vals, y=costs, mode='lines+markers', name="Op Cost"), row=1, col=1)
    fig.add_vline(x=n_opt_analytic, line_dash="dash", line_color="green", annotation_text="n_opt", row=1, col=1)
    fig.update_xaxes(title_text="Active Modules (n)", row=1, col=1)
    fig.update_yaxes(title_text="Cost (EUR/s)", row=1, col=1)

    # --- PLOT 2: The Temporal Hurdle vs Markov Time ---
    base_cost = cost_matrix[idx_p, idx_n0]
    req_times = []
    
    # Extract the exact E[T_hold] for our current specific load and state
    markov_time_for_state = E_t_hold_matrix[idx_p, idx_n0] 

    for j, n_cand in enumerate(n_vals):
        if n_cand == current_n:
            req_times.append(0)
            continue
            
        cand_cost = cost_matrix[idx_p, j]
        savings = base_cost - cand_cost
        
        if savings <= 0:
            req_times.append(0)
        else:
            penalty = abs(n_cand - current_n) * k_s
            req_times.append(penalty / savings)

    marker_colors = ['rgba(200, 200, 200, 0.5)' if t == 0 else 'red' if t > markov_time_for_state else 'green' for t in req_times]
    fig.add_trace(go.Bar(x=n_vals, y=req_times, marker_color=marker_colors, name="Req. Time"), row=1, col=2)
    fig.add_hline(y=markov_time_for_state, line_color="black", annotation_text=f"Markov E[T_hold]={markov_time_for_state:.0f}s", row=1, col=2)
    fig.update_yaxes(title_text="Required Time (s)", range=[0, min(20000, max(req_times)+1000)], row=1, col=2)

    # --- PLOT 3: The Decision Heatmap ---
    decision_matrix = np.zeros((len(n_vals), len(p_vals)))
    
    for l, p in enumerate(p_vals):
        ideal_j = np.argmin(cost_matrix[l, :])
        n_ideal = n_vals[ideal_j]
        
        for j, n in enumerate(n_vals):
            c_current = cost_matrix[l, j]
            t_hold = E_t_hold_matrix[l, j]
            
            if heuristic_mode == 'Discrete Search':
                c_target = cost_matrix[l, ideal_j]
                savings = c_current - c_target
                penalty = abs(n_ideal - n) * k_s
                if (savings * t_hold) > penalty:
                    decision_matrix[j, l] = 1 if n_ideal > n else -1
                    
            elif heuristic_mode == 'Analytical Step':
                step_dir = 0
                if n_ideal > n and j < len(n_vals)-1:
                    step_dir = 1
                elif n_ideal < n and j > 0:
                    step_dir = -1
                
                if step_dir != 0:
                    c_target = cost_matrix[l, j + step_dir]
                    savings = c_current - c_target
                    if (savings * t_hold) > k_s:  # single step penalty
                        decision_matrix[j, l] = step_dir

    colorscale = [[0.0, "red"], [0.5, "lightgrey"], [1.0, "green"]]
    fig.add_trace(go.Heatmap(z=decision_matrix, x=p_vals, y=n_vals, colorscale=colorscale, showscale=False), row=1, col=3)
    fig.update_xaxes(title_text="Demand P_d (kW)", row=1, col=3)
    fig.update_yaxes(title_text="Current Modules", row=1, col=3)

    fig.update_layout(height=550, width=1300, showlegend=False)
    fig.show()

# --- 5. UI BINDINGS ---
# Assuming days 4 through 14 are your available training files based on your script
train_days = [4, 5, 6, 7, 8, 9, 10, 11, 12, 14]

ui = widgets.interactive(
    render_dashboard,
    p_d_eval=widgets.FloatSlider(value=1000.0, min=np.min(config.p_vals), max=np.max(config.p_vals), step=10, description="P_d eval"),
    current_n=widgets.IntSlider(value=10, min=1, max=20, step=1, description="n(t-1)"),
    k_fc=widgets.IntSlider(value=config.PHYSICS.k_fc, min=30000, max=100000, step=5000, description="k_fc (EUR)"),
    s_max=widgets.IntSlider(value=config.PHYSICS.s_max, min=1000, max=10000, step=500, description="S_max"),
    training_day=widgets.Dropdown(options=train_days, value=5, description="Train Day"),
    tolerance=widgets.IntSlider(value=1, min=1, max=10, step=1, description="Tolerance"),
    heuristic_mode=widgets.Dropdown(options=['Discrete Search', 'Analytical Step'], description="Heuristic")
)

display(ui)

ModuleNotFoundError: No module named 'ipywidgets'